# 2DGS Mitsubishi Pipeline v2 — 두 경로 실험

`pipeline_diagnosis.ipynb` 결과로 확인된 **3.9_blender_path PSNR=45.58** 을 30k iter full run 으로 확장 + COLMAP 경로 수정판도 병렬 실험.

## 실험 구성

| ID | 경로 | 배경 | L1 loss | λ_dist | 초기 PCD | 예상 PSNR |
|---|---|---|---|---|---|---|
| **A**: `blender_path` | transforms JSON | white | plain | 0.0 | random 100k | **50+** (3.9 확장) |
| **B**: `colmap_fixed` | COLMAP binary | white | plain | 0.0 | GT-depth 50k | 35~45 |
| **C** (참고): `colmap_original` | COLMAP binary | black | mask-aware | 0.1 | GT-depth 8k | 13 (원본) |

**실험 A** 는 diagnosis 3.9 를 30k iter 로 연장 (검증된 경로).
**실험 B** 는 diagnosis 결과에서 원인으로 지목된 4가지 — RGBA→white RGB composite, mask-aware L1 제거, λ_dist=0, random init — 을 적용한 COLMAP 경로.
**실험 C** 는 대조군.

모든 결과는 `/content/results_v2/` → zip → Drive → 로컬.

## Part 0. 셋업

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!nvcc --version | tail -1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_SAVE = '/content/drive/MyDrive/2dgs_v2'
import os
os.makedirs(DRIVE_SAVE, exist_ok=True)
print(f'Drive: {DRIVE_SAVE}')

In [ ]:
import os
ROOT = '/content'
REPO = os.path.join(ROOT, '2d-gaussian-splatting')
RESULTS = os.path.join(ROOT, 'results_v2')
DATA_A = os.path.join(ROOT, 'data_A_blender')
DATA_B = os.path.join(ROOT, 'data_B_colmap_fixed')
DATA_C = os.path.join(ROOT, 'data_C_colmap_original')
for d in [RESULTS]: os.makedirs(d, exist_ok=True)
print('Dirs ready.')

In [ ]:
if not os.path.exists(REPO):
    !git clone https://github.com/BAEJUNHAK/2d-gaussian-splatting.git --recursive {REPO}

In [ ]:
# 의존성
!pip install -q plyfile scikit-image==0.21.0 lpips==0.1.4 trimesh==4.3.2 \
    open3d mediapy==1.1.2 opencv-python tqdm pillow tensorboard

# simple_knn FLT_MAX 패치
knn_cu = os.path.join(REPO, 'submodules', 'simple-knn', 'simple_knn.cu')
with open(knn_cu) as f: src = f.read()
if '#include <cfloat>' not in src:
    src = src.replace('#include "simple_knn.h"', '#include "simple_knn.h"\n#include <cfloat>')
    with open(knn_cu, 'w') as f: f.write(src)
    print('Patched simple_knn.cu')

# ⚠ train.py 는 plain L1 (저자 원본) 그대로 사용 — 실험 A, B 는 mask-aware 패치 없음
# 실험 C 만을 위해 나중에 토글 함수를 정의

!pip install -q {REPO}/submodules/diff-surfel-rasterization
!pip install -q {REPO}/submodules/simple-knn

In [ ]:
# 데이터 다운로드
!pip install -q gdown
import gdown, zipfile, numpy as np
RAW = os.path.join(ROOT, 'raw_data')
os.makedirs(RAW, exist_ok=True)
rgb_zip = os.path.join(RAW, 'dataset.zip')
if not os.path.exists(rgb_zip):
    gdown.download(id='1dnj1s-mqIuS6OcdSr5CczBr9u8yBzUUB', output=rgb_zip, quiet=False)
depth_zip = os.path.join(RAW, 'depth_data.zip')
if not os.path.exists(depth_zip):
    gdown.download(id='1KmXCzBYv_mkPmZnWka1ThCZSNHTyivKQ', output=depth_zip, quiet=False)

RGB_DIR = os.path.join(RAW, 'dataset')
DEPTH_DIR = os.path.join(RAW, 'dataset_depth')
if not os.path.exists(RGB_DIR):
    with zipfile.ZipFile(rgb_zip) as z: z.extractall(RAW)
if not os.path.exists(DEPTH_DIR):
    with zipfile.ZipFile(depth_zip) as z: z.extractall(RAW)

all_rgb = sorted(f for f in os.listdir(RGB_DIR) if f.startswith('rgb_'))
all_calib = sorted(f for f in os.listdir(RGB_DIR) if f.startswith('calib_'))
all_depth = sorted(f for f in os.listdir(DEPTH_DIR) if f.startswith('depth_'))

NUM_VIEWS = 50
indices = np.linspace(0, len(all_rgb) - 1, NUM_VIEWS, dtype=int)
rgb_files = [all_rgb[i] for i in indices]
calib_files = [all_calib[i] for i in indices]
depth_files = [all_depth[i] for i in indices]
print(f'Sampled {NUM_VIEWS} views.')

In [ ]:
# 공통 유틸
import struct, shutil, math, json
from configparser import ConfigParser
from PIL import Image

def parse_calib(filepath):
    with open(filepath) as f:
        lines = [l for l in f.readlines() if not l.startswith('#')]
    cp = ConfigParser(); cp.read_string('\n'.join(lines)); sec = 'camera_0'
    K = np.array(list(map(float, cp.get(sec,'k_matrix').split('Matrix:3:3:')[1].split(',')))).reshape(3,3)
    R = np.array(list(map(float, cp.get(sec,'r_matrix').split('Matrix:3:3:')[1].split(',')))).reshape(3,3)
    t = np.array(list(map(float, cp.get(sec,'t_vector').split('Vector:3:')[1].split(','))))
    w = int(cp.get(sec,'width')); h = int(cp.get(sec,'height'))
    return K, R, t, w, h

def rotmat2qvec(R):
    Rxx,Ryx,Rzx,Rxy,Ryy,Rzy,Rxz,Ryz,Rzz = R.flat
    K_ = np.array([[Rxx-Ryy-Rzz,0,0,0],[Ryx+Rxy,Ryy-Rxx-Rzz,0,0],
                   [Rzx+Rxz,Rzy+Ryz,Rzz-Rxx-Ryy,0],
                   [Ryz-Rzy,Rzx-Rxz,Rxy-Ryx,Rxx+Ryy+Rzz]]) / 3.0
    ev, vec = np.linalg.eigh(K_)
    q = vec[[3,0,1,2], np.argmax(ev)]
    if q[0] < 0: q *= -1
    return q

def write_cameras_bin(cam_dict, path):
    with open(path,'wb') as f:
        f.write(struct.pack('<Q', len(cam_dict)))
        for cid, c in cam_dict.items():
            f.write(struct.pack('<iiQQ', cid, c['model_id'], c['width'], c['height']))
            for p in c['params']: f.write(struct.pack('<d', p))

def write_images_bin(img_dict, path):
    with open(path,'wb') as f:
        f.write(struct.pack('<Q', len(img_dict)))
        for iid, img in img_dict.items():
            f.write(struct.pack('<i', iid))
            for q in img['qvec']: f.write(struct.pack('<d', q))
            for t in img['tvec']: f.write(struct.pack('<d', t))
            f.write(struct.pack('<i', img['camera_id']))
            f.write(img['name'].encode()); f.write(b'\x00')
            f.write(struct.pack('<Q', 0))

def write_points3D_bin(pts_dict, path):
    with open(path,'wb') as f:
        f.write(struct.pack('<Q', len(pts_dict)))
        for pid, p in pts_dict.items():
            f.write(struct.pack('<Q', pid))
            for x in p['xyz']: f.write(struct.pack('<d', x))
            for c in p['rgb']: f.write(struct.pack('<B', int(c)))
            f.write(struct.pack('<d', p['error']))
            f.write(struct.pack('<Q', 0))

SCALE = 100.0
print('Utils ready.')

## Part 1. 실험 A — Blender JSON 경로 (3.9 확장)

`transforms_train.json` / `transforms_test.json` 생성 → `readNerfSyntheticInfo` 가 자동으로
- RGBA 를 white 배경에 composite
- random 100k 포인트로 초기화

llffhold=8 대신 train/test 디렉토리 분리 방식 사용 (저자 Blender 표준).

In [ ]:
os.makedirs(DATA_A, exist_ok=True)
os.makedirs(os.path.join(DATA_A, 'train'), exist_ok=True)
os.makedirs(os.path.join(DATA_A, 'test'), exist_ok=True)

K0, _, _, w0, h0 = parse_calib(os.path.join(RGB_DIR, calib_files[0]))
fx0 = K0[0,0]
fov_x = float(2*math.atan(w0/(2*fx0)))

train_frames, test_frames = [], []
for i, (rgb_f, cal_f, dep_f) in enumerate(zip(rgb_files, calib_files, depth_files)):
    _, R_raw, t_raw, _, _ = parse_calib(os.path.join(RGB_DIR, cal_f))
    # H1 (R,t = w2c) → c2w
    R_w2c = R_raw; t_w2c = t_raw / SCALE
    R_c2w = R_w2c.T; t_c2w = -R_w2c.T @ t_w2c
    c2w = np.eye(4); c2w[:3,:3] = R_c2w; c2w[:3,3] = t_c2w
    # readCamerasFromTransforms 가 c2w[:3,1:3]*=-1 을 적용하므로 미리 역으로 뒤집어 원상복구
    c2w[:3, 1:3] *= -1
    split = 'test' if i % 8 == 0 else 'train'
    dest = os.path.join(DATA_A, split, os.path.basename(rgb_f))
    # 원본 RGB + GT depth 로 RGBA 저장 (readNerfSyntheticInfo 가 composite)
    rgb_arr = np.array(Image.open(os.path.join(RGB_DIR, rgb_f)))[..., :3]
    depth_arr = np.array(Image.open(os.path.join(DEPTH_DIR, dep_f)))
    bg = depth_arr == 0
    rgb_arr[bg] = 0
    alpha = ((~bg) * 255).astype(np.uint8)
    rgba = np.concatenate([rgb_arr, alpha[..., None]], axis=-1)
    Image.fromarray(rgba, 'RGBA').save(dest)
    frame = {'file_path': f'./{split}/' + os.path.basename(rgb_f).replace('.png',''),
             'transform_matrix': c2w.tolist()}
    (test_frames if split=='test' else train_frames).append(frame)

for split, frames in [('train', train_frames), ('test', test_frames)]:
    with open(os.path.join(DATA_A, f'transforms_{split}.json'), 'w') as f:
        json.dump({'camera_angle_x': fov_x, 'frames': frames}, f, indent=2)
print(f'DATA_A: train={len(train_frames)}, test={len(test_frames)}')

## Part 2. 실험 B — COLMAP 수정판 (white composite + plain L1 + no reg + 50k init)

COLMAP 경로를 유지하되 diagnosis 에서 지목된 4가지 문제를 모두 해결:
1. ✅ RGB 를 **white 와 composite 한 3채널** 로 저장 (RGBA 아님 → loadCam 에서 gt_alpha_mask=None)
2. ✅ mask-aware L1 패치 **제거** (train.py 원본 유지)
3. ✅ `--lambda_dist 0.0`
4. ✅ 초기 PCD 를 **50000 점** 으로 증가 (뷰당 2000 → 1000점, stride=1 전뷰 sampling)

In [ ]:
os.makedirs(os.path.join(DATA_B, 'images'), exist_ok=True)
os.makedirs(os.path.join(DATA_B, 'sparse', '0'), exist_ok=True)

cams = {1:{'model_id':1,'width':w0,'height':h0,'params':[K0[0,0],K0[1,1],K0[0,2],K0[1,2]]}}
write_cameras_bin(cams, os.path.join(DATA_B, 'sparse','0','cameras.bin'))

# 이미지: white 배경 composite, 3채널 RGB 저장
imgs_b = {}
for i, (rgb_f, cal_f, dep_f) in enumerate(zip(rgb_files, calib_files, depth_files)):
    dst = os.path.join(DATA_B, 'images', rgb_f)
    if not os.path.exists(dst):
        rgb_arr = np.array(Image.open(os.path.join(RGB_DIR, rgb_f)))[..., :3].astype(np.float32)
        depth_arr = np.array(Image.open(os.path.join(DEPTH_DIR, dep_f)))
        bg = depth_arr == 0
        # composite against white
        rgb_arr[bg] = 255.0
        Image.fromarray(rgb_arr.astype(np.uint8), 'RGB').save(dst)
    _, R_raw, t_raw, _, _ = parse_calib(os.path.join(RGB_DIR, cal_f))
    imgs_b[i+1] = {'qvec': rotmat2qvec(R_raw), 'tvec': t_raw/SCALE, 'camera_id':1, 'name':rgb_f}
write_images_bin(imgs_b, os.path.join(DATA_B, 'sparse','0','images.bin'))

# 초기 PCD: 50뷰 전체에서 뷰당 1000점 → 최대 50k (voxel downsampling 후)
from tqdm import tqdm
all_pts, all_cols = [], []
fx, fy, cx, cy = K0[0,0], K0[1,1], K0[0,2], K0[1,2]
for idx in tqdm(range(len(rgb_files)), desc='PCD'):
    depth_raw = np.array(Image.open(os.path.join(DEPTH_DIR, depth_files[idx]))).astype(np.float64)
    depth = depth_raw * 0.01 / SCALE
    rgb_arr = np.array(Image.open(os.path.join(RGB_DIR, rgb_files[idx])))[..., :3]
    fg_v, fg_u = np.where(depth_raw > 0)
    if len(fg_v) == 0: continue
    n = min(1000, len(fg_v))
    ch = np.random.RandomState(idx).choice(len(fg_v), n, replace=False)
    u = fg_u[ch].astype(np.float64); v = fg_v[ch].astype(np.float64)
    z = depth[fg_v[ch], fg_u[ch]]
    x = (u-cx)*z/fx; y = (v-cy)*z/fy
    pts_cam = np.stack([x,y,z], axis=1)
    _, R_raw, t_raw, _, _ = parse_calib(os.path.join(RGB_DIR, calib_files[idx]))
    t_norm = t_raw / SCALE
    pts_world = (R_raw.T @ (pts_cam - t_norm).T).T
    cols = rgb_arr[fg_v[ch], fg_u[ch], :3]
    all_pts.append(pts_world); all_cols.append(cols)
pts = np.concatenate(all_pts); cols = np.concatenate(all_cols)
vox = 0.005
coords = np.floor(pts/vox).astype(np.int64)
_, uq = np.unique(coords, axis=0, return_index=True)
pts, cols = pts[uq], cols[uq]
pts_dict = {i+1:{'xyz':pts[i], 'rgb':cols[i], 'error':0.0} for i in range(len(pts))}
write_points3D_bin(pts_dict, os.path.join(DATA_B, 'sparse','0','points3D.bin'))
print(f'DATA_B: {len(pts)} init points, range [{pts.min(0)}] ~ [{pts.max(0)}]')

## Part 3. 실험 C — 원본 COLMAP (대조군, 기존 Colab 세팅)

In [ ]:
os.makedirs(os.path.join(DATA_C, 'images'), exist_ok=True)
os.makedirs(os.path.join(DATA_C, 'sparse', '0'), exist_ok=True)
write_cameras_bin(cams, os.path.join(DATA_C, 'sparse','0','cameras.bin'))

# RGBA 저장 (alpha=mask)
imgs_c = {}
for i, (rgb_f, cal_f, dep_f) in enumerate(zip(rgb_files, calib_files, depth_files)):
    dst = os.path.join(DATA_C, 'images', rgb_f)
    if not os.path.exists(dst):
        rgb_arr = np.array(Image.open(os.path.join(RGB_DIR, rgb_f)))
        depth_arr = np.array(Image.open(os.path.join(DEPTH_DIR, dep_f)))
        bg = depth_arr == 0
        rgb_arr[bg] = 0
        alpha = ((~bg) * 255).astype(np.uint8)
        rgba = np.concatenate([rgb_arr[..., :3], alpha[..., None]], axis=-1)
        Image.fromarray(rgba, 'RGBA').save(dst)
    _, R_raw, t_raw, _, _ = parse_calib(os.path.join(RGB_DIR, cal_f))
    imgs_c[i+1] = {'qvec': rotmat2qvec(R_raw), 'tvec': t_raw/SCALE, 'camera_id':1, 'name':rgb_f}
write_images_bin(imgs_c, os.path.join(DATA_C, 'sparse','0','images.bin'))

# 원본 Colab 과 동일한 8k 포인트 초기화 (stride=10, 2000pts/view)
all_pts, all_cols = [], []
for idx in range(0, len(rgb_files), 10):
    depth_raw = np.array(Image.open(os.path.join(DEPTH_DIR, depth_files[idx]))).astype(np.float64)
    depth = depth_raw * 0.01 / SCALE
    rgb_arr = np.array(Image.open(os.path.join(RGB_DIR, rgb_files[idx])))[..., :3]
    fg_v, fg_u = np.where(depth_raw > 0)
    if len(fg_v) == 0: continue
    n = min(2000, len(fg_v))
    ch = np.random.RandomState(idx).choice(len(fg_v), n, replace=False)
    u = fg_u[ch].astype(np.float64); v = fg_v[ch].astype(np.float64)
    z = depth[fg_v[ch], fg_u[ch]]
    x = (u-cx)*z/fx; y = (v-cy)*z/fy
    pts_cam = np.stack([x,y,z], axis=1)
    _, R_raw, t_raw, _, _ = parse_calib(os.path.join(RGB_DIR, calib_files[idx]))
    t_norm = t_raw / SCALE
    pts_world = (R_raw.T @ (pts_cam - t_norm).T).T
    cols = rgb_arr[fg_v[ch], fg_u[ch], :3]
    all_pts.append(pts_world); all_cols.append(cols)
pts = np.concatenate(all_pts); cols = np.concatenate(all_cols)
coords = np.floor(pts/0.01).astype(np.int64)
_, uq = np.unique(coords, axis=0, return_index=True)
pts, cols = pts[uq], cols[uq]
pts_dict = {i+1:{'xyz':pts[i], 'rgb':cols[i], 'error':0.0} for i in range(len(pts))}
write_points3D_bin(pts_dict, os.path.join(DATA_C, 'sparse','0','points3D.bin'))
print(f'DATA_C: {len(pts)} init points')

## Part 4. train.py 패치 토글 (실험 C 용 mask-aware L1)

In [ ]:
train_py = os.path.join(REPO, 'train.py')
with open(train_py) as f: _orig = f.read()

_orig_block = '''        gt_image = viewpoint_cam.original_image.cuda()
        Ll1 = l1_loss(image, gt_image)
        loss = (1.0 - opt.lambda_dssim) * Ll1 + opt.lambda_dssim * (1.0 - ssim(image, gt_image))'''

_patched_block = '''        gt_image = viewpoint_cam.original_image.cuda()
        gt_alpha_mask = viewpoint_cam.gt_alpha_mask
        if gt_alpha_mask is not None:
            gt_alpha_mask = gt_alpha_mask.cuda()
            diff = torch.abs(image - gt_image).mean(0, keepdim=True)
            Ll1 = (diff * gt_alpha_mask).sum() / (gt_alpha_mask.sum() + 1e-8)
        else:
            Ll1 = l1_loss(image, gt_image)
        loss = (1.0 - opt.lambda_dssim) * Ll1 + opt.lambda_dssim * (1.0 - ssim(image, gt_image))'''

def set_mask_patch(enable):
    with open(train_py) as f: src = f.read()
    if enable:
        if _orig_block in src: src = src.replace(_orig_block, _patched_block)
    else:
        if _patched_block in src: src = src.replace(_patched_block, _orig_block)
    with open(train_py,'w') as f: f.write(src)

# 기본 상태: 원본 (plain L1)
set_mask_patch(False)
print('train.py in plain L1 mode (author original).')

## Part 5. 학습 (30k iter × 3 실험)
총 ~60분 예상 (GPU, 각 실험 ~20분).

In [ ]:
OUT_A = os.path.join(ROOT, 'output_A')
OUT_B = os.path.join(ROOT, 'output_B')
OUT_C = os.path.join(ROOT, 'output_C')

In [ ]:
# 실험 A: Blender JSON 경로
set_mask_patch(False)
!rm -rf {OUT_A}
!cd {REPO} && python train.py \
    -s {DATA_A} -m {OUT_A} --eval \
    --white_background \
    --sh_degree 1 --lambda_normal 0.0 --lambda_dist 0.0 \
    --depth_ratio 1.0 --iterations 30000 \
    --test_iterations 7000 15000 30000 \
    --save_iterations 7000 30000 \
    --checkpoint_iterations 30000

In [ ]:
# 실험 B: COLMAP 수정판
set_mask_patch(False)
!rm -rf {OUT_B}
!cd {REPO} && python train.py \
    -s {DATA_B} -m {OUT_B} --eval \
    --white_background \
    --sh_degree 1 --lambda_normal 0.0 --lambda_dist 0.0 \
    --depth_ratio 1.0 --iterations 30000 \
    --test_iterations 7000 15000 30000 \
    --save_iterations 7000 30000 \
    --checkpoint_iterations 30000

In [ ]:
# 실험 C: 원본 Colab 세팅 (대조군)
set_mask_patch(True)   # mask-aware L1 복원
!rm -rf {OUT_C}
!cd {REPO} && python train.py \
    -s {DATA_C} -m {OUT_C} --eval \
    --sh_degree 1 --lambda_normal 0.0 --lambda_dist 0.1 \
    --depth_ratio 1.0 --iterations 30000 \
    --test_iterations 7000 15000 30000 \
    --save_iterations 7000 30000 \
    --checkpoint_iterations 30000
# 복원
set_mask_patch(False)

## Part 6. 렌더링 + 메트릭

In [ ]:
# 모든 실험 test 뷰 렌더 + metrics
for name, out_dir, data_dir in [('A', OUT_A, DATA_A), ('B', OUT_B, DATA_B), ('C', OUT_C, DATA_C)]:
    print(f'\n===== RENDER {name} =====')
    !cd {REPO} && python render.py -m {out_dir} -s {data_dir} \
        --depth_ratio 1.0 --skip_train --skip_mesh 2>&1 | tail -5
    print(f'\n===== METRICS {name} =====')
    !cd {REPO} && python metrics.py -m {out_dir} 2>&1 | tail -10

## Part 7. 메쉬 추출

In [ ]:
# bounded TSDF fusion
for name, out_dir, data_dir in [('A', OUT_A, DATA_A), ('B', OUT_B, DATA_B), ('C', OUT_C, DATA_C)]:
    print(f'\n===== MESH {name} =====')
    !cd {REPO} && python render.py -m {out_dir} -s {data_dir} \
        --depth_ratio 1.0 --skip_train --skip_test --num_cluster 50 2>&1 | tail -8

## Part 8. 결과 집계

In [ ]:
import glob
import matplotlib.pyplot as plt

summary = {}
for name, out_dir in [('A_blender', OUT_A), ('B_colmap_fixed', OUT_B), ('C_colmap_original', OUT_C)]:
    entry = {}
    res_path = os.path.join(out_dir, 'results.json')
    if os.path.exists(res_path):
        with open(res_path) as f: r = json.load(f)
        if r:
            mk = list(r.keys())[0]
            entry.update(r[mk])
    # 포인트 수
    pcd_paths = sorted(glob.glob(os.path.join(out_dir, 'point_cloud', 'iteration_*')))
    if pcd_paths:
        try:
            from plyfile import PlyData
            pd = PlyData.read(os.path.join(pcd_paths[-1], 'point_cloud.ply'))
            entry['num_gaussians'] = len(pd['vertex'])
        except: pass
    summary[name] = entry

# 결과 디렉토리에 복사
for name, out_dir in [('A_blender', OUT_A), ('B_colmap_fixed', OUT_B), ('C_colmap_original', OUT_C)]:
    dst = os.path.join(RESULTS, name)
    os.makedirs(dst, exist_ok=True)
    for fn in ['results.json', 'per_view.json', 'cfg_args']:
        src = os.path.join(out_dir, fn)
        if os.path.exists(src): shutil.copy2(src, dst)
    # 첫 3 test renders + 1 gt
    test_renders = sorted(glob.glob(os.path.join(out_dir, 'test', 'ours_*', 'renders', '*.png')))[:3]
    test_gts = sorted(glob.glob(os.path.join(out_dir, 'test', 'ours_*', 'gt', '*.png')))[:1]
    for f in test_renders: shutil.copy2(f, dst)
    for f in test_gts: shutil.copy2(f, os.path.join(dst, 'gt_' + os.path.basename(f)))
    # mesh
    for mesh_name in ['fuse.ply', 'fuse_post.ply']:
        mp = sorted(glob.glob(os.path.join(out_dir, 'train', 'ours_*', mesh_name)))
        if mp: shutil.copy2(mp[-1], dst)
    # depth tiff 1장
    depths = sorted(glob.glob(os.path.join(out_dir, 'test', 'ours_*', 'vis', 'depth_*.tiff')))[:1]
    for f in depths: shutil.copy2(f, dst)

with open(os.path.join(RESULTS, 'summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

In [ ]:
# 비교 차트
names = list(summary.keys())
psnrs = [summary[n].get('PSNR', 0) for n in names]
ssims = [summary[n].get('SSIM', 0) for n in names]
lpipss = [summary[n].get('LPIPS', 0) for n in names]
gs_counts = [summary[n].get('num_gaussians', 0) for n in names]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
ax = axes[0,0]
ax.bar(names, psnrs, color=['seagreen','steelblue','tomato'])
ax.axhline(y=12.88, color='gray', linestyle='--', label='Original Colab 30k (12.88)')
for i, v in enumerate(psnrs):
    ax.text(i, v+0.5, f'{v:.2f}', ha='center')
ax.set_title('PSNR @ 30k iter'); ax.legend()

ax = axes[0,1]
ax.bar(names, ssims, color=['seagreen','steelblue','tomato'])
for i, v in enumerate(ssims):
    ax.text(i, v+0.01, f'{v:.4f}', ha='center')
ax.set_title('SSIM'); ax.set_ylim(0, 1.05)

ax = axes[1,0]
ax.bar(names, lpipss, color=['seagreen','steelblue','tomato'])
for i, v in enumerate(lpipss):
    ax.text(i, v+0.005, f'{v:.4f}', ha='center')
ax.set_title('LPIPS (lower=better)')

ax = axes[1,1]
ax.bar(names, gs_counts, color=['seagreen','steelblue','tomato'])
for i, v in enumerate(gs_counts):
    ax.text(i, v+1000, f'{v}', ha='center')
ax.set_title('# Gaussians')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS, 'comparison_charts.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 3x4 렌더 콜라주 (3 실험 × GT + 3 test renders)
fig, axes = plt.subplots(3, 4, figsize=(18, 13))
for row, (name, out_dir) in enumerate([('A_blender', OUT_A), ('B_colmap_fixed', OUT_B), ('C_colmap_original', OUT_C)]):
    gts = sorted(glob.glob(os.path.join(out_dir, 'test', 'ours_*', 'gt', '*.png')))
    rs = sorted(glob.glob(os.path.join(out_dir, 'test', 'ours_*', 'renders', '*.png')))
    if gts:
        axes[row, 0].imshow(np.array(Image.open(gts[0])))
        axes[row, 0].set_title(f'{name}\nGT')
    for j in range(3):
        if j < len(rs):
            axes[row, j+1].imshow(np.array(Image.open(rs[j])))
            axes[row, j+1].set_title(f'render v{j}')
    for j in range(4):
        axes[row, j].axis('off')
    axes[row, 0].text(-50, 400, f"PSNR={summary[name].get('PSNR',0):.2f}",
                      fontsize=14, fontweight='bold', rotation=90, va='center')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS, 'render_collage.png'), dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# summary.md
def fmt(v, spec='.4f'):
    return format(v, spec) if isinstance(v, (int, float)) else 'NA'

lines = ['# 2DGS Pipeline v2 — 두 경로 실험 결과\n\n']
lines.append('| Experiment | PSNR | SSIM | LPIPS | #Gauss |\n|---|---|---|---|---|\n')
for n in names:
    lines.append('| {} | {} | {} | {} | {} |\n'.format(
        n, fmt(summary[n].get('PSNR'), '.2f'),
        fmt(summary[n].get('SSIM')),
        fmt(summary[n].get('LPIPS')),
        summary[n].get('num_gaussians', 'NA')))
best = max(names, key=lambda n: summary[n].get('PSNR', 0))
lines.append('\n**Best**: {} (PSNR={})\n'.format(best, fmt(summary[best].get('PSNR'), '.2f')))
lines.append('\n**Original Colab 30k baseline**: PSNR=12.88\n')
with open(os.path.join(RESULTS, 'summary.md'), 'w') as f:
    f.writelines(lines)
print(''.join(lines))

## Part 9. Drive 백업

In [ ]:
ZIP_PATH = '/tmp/results_v2.zip'
!rm -f {ZIP_PATH}
!cd /content && zip -qr {ZIP_PATH} results_v2
!ls -lh {ZIP_PATH}
!cp {ZIP_PATH} {DRIVE_SAVE}/
# 체크포인트도 Drive 에 백업 (용량 큼)
for n, out_dir in [('A', OUT_A), ('B', OUT_B), ('C', OUT_C)]:
    !cp -r {out_dir}/point_cloud {DRIVE_SAVE}/ckpt_{n}_point_cloud 2>/dev/null
    !cp {out_dir}/cfg_args {DRIVE_SAVE}/ckpt_{n}_cfg_args 2>/dev/null
print(f'Uploaded: {DRIVE_SAVE}/results_v2.zip')

## Part 10. 로컬 다운로드 지침

```bash
# Drive 웹에서 2dgs_v2/results_v2.zip 다운로드 후
cd /Users/baejunhak/labatory/2d-gaussian-splatting
unzip -o ~/Downloads/results_v2.zip
```

그 다음 Claude 에게 `/Users/baejunhak/labatory/2d-gaussian-splatting/results_v2 읽어` 하시면 됩니다.

**핵심 파일**:
- `results_v2/summary.md`
- `results_v2/comparison_charts.png`
- `results_v2/render_collage.png`
- `results_v2/A_blender/fuse_post.ply` (Best mesh)
- `results_v2/B_colmap_fixed/fuse_post.ply`
- `results_v2/C_colmap_original/fuse_post.ply`